# Covid-19 Data Visualization

Data provided by Johns Hopkins CSSE on [GitHub](https://github.com/CSSEGISandData/COVID-19)



### Define class to download and preprocess raw data

In [117]:
import pandas as pd


class Data():

    def get_data(self):
        self.dtf_cases = pd.read_csv(
            "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv", sep=",")
        self.dtf_deaths = pd.read_csv(
            "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv", sep=",")
        self.dtf_recovered = pd.read_csv(
            "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_recovered_global.csv", sep=",")
        self.countrylist = ["World"] + \
            self.dtf_cases["Country/Region"].unique().tolist()

    @staticmethod
    def group_by_country(dtf, country):
        dtf = dtf.drop(['Province/State', 'Lat', 'Long'],
                       axis=1).groupby("Country/Region").sum().T
        dtf["World"] = dtf.sum(axis=1)
        dtf = dtf[country]
        dtf.index = pd.to_datetime(dtf.index, infer_datetime_format=True)
        ts = pd.DataFrame(index=dtf.index, data=dtf.values, columns=["data"])
        return ts

    def process_data(self, country):
        # cumulated cases by country
        self.dtf = self.group_by_country(self.dtf_cases, country)

        # add deaths by country
        deaths = self.group_by_country(self.dtf_deaths, country)
        self.dtf["deaths"] = deaths
        
        # add recovered by country
        recovered = self.group_by_country(self.dtf_recovered, country)
        self.dtf["recovered"] = recovered
        
        # calculate cumulated still active cases: data - recovered - deaths
        active = self.dtf["data"] - self.dtf["recovered"] - self.dtf["deaths"]
        self.dtf["active"] = active

        # calculate daily new cases
        daily = self.dtf["data"] - self.dtf["data"].shift(1)  
        self.dtf["daily_cases"] = daily       
        self.dtf["daily_cases"] = self.dtf["daily_cases"].fillna(method='bfill') # fill Nas

        # remove data until first day with none zero cases
        idx = self.dtf["data"][self.dtf["data"]!=0].index[0]
        self.dtf = self.dtf.loc[idx:]       

### Initialize raw data

In [118]:
# Read data
data = Data()
data.get_data()

data.dtf_cases

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,5/4/20,5/5/20,5/6/20,5/7/20,5/8/20,5/9/20,5/10/20,5/11/20,5/12/20,5/13/20
0,NaN,Afghanistan,33.000000,65.000000,0,0,0,0,0,0,...,2894,3224,3392,3563,3778,4033,4402,4687,4963,5226
1,NaN,Albania,41.153300,20.168300,0,0,0,0,0,0,...,803,820,832,842,850,856,868,872,876,880
2,NaN,Algeria,28.033900,1.659600,0,0,0,0,0,0,...,4648,4838,4997,5182,5369,5558,5723,5891,6067,6253
3,NaN,Andorra,42.506300,1.521800,0,0,0,0,0,0,...,750,751,751,752,752,754,755,755,758,760
4,NaN,Angola,-11.202700,17.873900,0,0,0,0,0,0,...,35,36,36,36,43,43,45,45,45,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,NaN,Sao Tome and Principe,0.186360,6.613081,0,0,0,0,0,0,...,23,174,174,187,208,208,208,208,208,220
262,NaN,Yemen,15.552727,48.516388,0,0,0,0,0,0,...,12,22,25,25,34,34,51,56,65,70
263,NaN,Comoros,-11.645500,43.333300,0,0,0,0,0,0,...,3,3,8,8,8,11,11,11,11,11
264,NaN,Tajikistan,38.861034,71.276093,0,0,0,0,0,0,...,230,293,379,461,522,612,612,661,729,801


### Clean-up data and extract data for single country or world

In [119]:
from ipywidgets import interactive
from IPython.display import display

def f(x):
    return x

w = interactive(f, x=data.countrylist)
display(w)



interactive(children=(Dropdown(description='x', options=('World', 'Afghanistan', 'Albania', 'Algeria', 'Andorr…

In [120]:
country = w.result

country

'World'

In [121]:
data.process_data(country)

data.dtf

,data,deaths,recovered,active,daily_cases
2020-01-22,555,17,28,510,99.0
2020-01-23,654,18,30,606,99.0
2020-01-24,941,26,36,879,287.0
2020-01-25,1434,42,39,1353,493.0
2020-01-26,2118,56,52,2010,684.0
...,...,...,...,...,...
2020-05-09,4024009,279311,1375624,2369074,85945.0
2020-05-10,4101699,282709,1408980,2410010,77690.0
2020-05-11,4177502,286330,1456209,2434963,75803.0
2020-05-12,4261747,291942,1493414,2476391,84245.0


### Plot comulative raw data

In [122]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=data.dtf["deaths"],
    name='deaths',
    marker_color='firebrick',
    fill='tonexty'
))

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=data.dtf["recovered"],
    name='recovered',
    marker_color='darkcyan',
    fill='tonexty'
))

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=data.dtf["active"],
    name='active',
    marker_color='royalblue',
    fill='tonexty'
))

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=data.dtf["data"],
    name='confirmed',
    marker_color='slategrey',
    fill='tonexty'
))

# add slider
fig.update_xaxes(
    rangeslider_visible=True
)

# set title & background color
fig.update_layout(
    title="Cumulative data: " + country,
    template='plotly_dark'
)

### Plot relative data

In [123]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=100*data.dtf["deaths"]/data.dtf["data"],
    name='deaths',
    mode='lines',
    marker_color='firebrick',
    stackgroup='one'
))

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=100*data.dtf["recovered"]/data.dtf["data"],
    name='recovered',
    mode='lines',
    marker_color='darkcyan',
    stackgroup='one'
))

fig.add_trace(go.Scatter(
    x=data.dtf.index,
    y=100*data.dtf["active"]/data.dtf["data"],
    name='active',
    mode='lines',
    marker_color='royalblue',
    stackgroup='one'
))

# add slider
fig.update_xaxes(
    rangeslider_visible=True
)

# set title & background color
fig.update_layout(
    title="Relative data: " + country,
    template='plotly_dark',
    yaxis = dict(type = 'linear', range = [0, 100], ticksuffix = '% ')
)

### Define class to add simple exponential model fit

In [124]:
import numpy as np
import pandas as pd
from scipy import optimize


class Model():

    def __init__(self, dtf):
        self.dtf = dtf

    @staticmethod
    def f(X, c, k, m):
        y = c / (1 + np.exp(-k*(X-m)))
        return y

    @staticmethod
    def fit_parametric(X, y, f, p0):
        model, cov = optimize.curve_fit(f, X, y, maxfev=10000, p0=p0)
        return model

    @staticmethod
    def forecast_parametric(model, f, X):
        preds = f(X, model[0], model[1], model[2])
        return preds

    def apply_model(self):
        # fit commulative cases
        y = self.dtf["data"].values
        t = np.arange(len(y))
        model = self.fit_parametric(t, y, self.f, p0=[np.max(y), 1, 1])
        fitted = self.f(t, model[0], model[1], model[2])
        self.dtf["fit_data"] = fitted

        # add diff
        self.dtf["delta_fit_data"] = self.dtf["fit_data"] - self.dtf["fit_data"].shift(1)        
        self.dtf["delta_fit_data"] = self.dtf["delta_fit_data"].fillna(method='bfill') # fill Nas

### Apply fit 

In [125]:
model = Model(data.dtf)
model.apply_model()

model.dtf

,data,deaths,recovered,active,daily_cases,fit_data,delta_fit_data
2020-01-22,555,17,28,510,99.0,3.866630e+03,325.216102
2020-01-23,654,18,30,606,99.0,4.191847e+03,325.216102
2020-01-24,941,26,36,879,287.0,4.544390e+03,352.543358
2020-01-25,1434,42,39,1353,493.0,4.926552e+03,382.162285
2020-01-26,2118,56,52,2010,684.0,5.340816e+03,414.264262
...,...,...,...,...,...,...,...
2020-05-09,4024009,279311,1375624,2369074,85945.0,3.968170e+06,54655.266161
2020-05-10,4101699,282709,1408980,2410010,77690.0,4.019953e+06,51782.563419
2020-05-11,4177502,286330,1456209,2434963,75803.0,4.068927e+06,48974.585337
2020-05-12,4261747,291942,1493414,2476391,84245.0,4.115169e+06,46241.838254


### Plot daily cases + model fit 

In [126]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=model.dtf.index,
    y=model.dtf["daily_cases"],
    name='confirmed',
    marker_color='slategrey')
)

fig.add_trace(go.Scatter(
    x=model.dtf.index,
    y=model.dtf["delta_fit_data"],
    mode='none',
    name='fit',
    fill='tozeroy',
    fillcolor='rgba(140, 80, 180, 0.25)'
))

# add slider
fig.update_xaxes(
    rangeslider_visible=True
)

# set background color
fig.update_layout(
    title="Daily cases: " + country,
    template='plotly_dark'
)

### Plot cumulative cases + model fit 

In [127]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=model.dtf.index,
    y=model.dtf["data"],
    mode='markers',
    name='confirmed',
    marker_color='slategrey'
))

fig.add_trace(go.Scatter(
    x=model.dtf.index,
    y=model.dtf["fit_data"],
    mode='none',
    name='fit',
    fill='tozeroy',
    fillcolor='rgba(140, 80, 180, 0.25)'
))

fig.add_trace(go.Bar(
    x=model.dtf.index,
    y=model.dtf["active"],
    name='active',
    marker_color='royalblue'
))

fig.add_trace(go.Bar(
    x=model.dtf.index,
    y=model.dtf["recovered"],
    name='recovered',
    marker_color='darkcyan'
))

fig.add_trace(go.Bar(
    x=model.dtf.index,
    y=model.dtf["deaths"],
    name='deaths',
    marker_color='firebrick'
))

# add slider
fig.update_xaxes(
    rangeslider_visible=True
)

# set background color
fig.update_layout(
    title="Extrapolation of cumulative data: " + country,
    template='plotly_dark'
)

### "Ferroic countries"

In [128]:
countries = ["Spain", "France", "Germany", "Russia", "China", "India", "Switzerland", "Sweden", "Ukraine", "Colombia", "Croatia", "Lithuania", "Taiwan*"]

dtf_cases = data.dtf_cases.drop(['Province/State', 'Lat', 'Long'],
                       axis=1).groupby("Country/Region").sum().T[countries]

dtf_cases

Country/Region,Spain,France,Germany,Russia,China,India,Switzerland,Sweden,Ukraine,Colombia,Croatia,Lithuania,Taiwan*
1/22/20,0,0,0,0,548,0,0,0,0,0,0,0,1
1/23/20,0,0,0,0,643,0,0,0,0,0,0,0,1
1/24/20,0,2,0,0,920,0,0,0,0,0,0,0,3
1/25/20,0,3,0,0,1406,0,0,0,0,0,0,0,3
1/26/20,0,3,0,0,2075,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5/9/20,223578,176782,171324,198676,83990,62808,30251,25921,14710,10495,2176,1444,440
5/10/20,224350,177094,171879,209688,84010,67161,30305,26322,15232,11063,2187,1479,440
5/11/20,227436,177547,172576,221344,84011,70768,30344,26670,15648,11613,2196,1485,440
5/12/20,228030,178349,173171,232243,84018,74292,30380,27272,16023,12272,2207,1491,440


In [129]:
fig = go.Figure()

for idx, country in enumerate(countries):
    fig.add_trace(go.Scatter(
        x=dtf_cases.index,
        y=dtf_cases[country],
        name=country
    ))

# add linear/log toggle button
fig.update_layout(    
    updatemenus = [
        dict(type="buttons",
             showactive=False,
             direction="right",
             buttons=list([            
                dict(label='lin/log',
                    method='relayout',
                    args=[{'yaxis': {'type': 'linear'}}],
                    args2=[{'yaxis': {'type': 'log'}}])
            ]),
            x=0.17,
            xanchor="left",
            y=1.095,
            yanchor="top"
        )
    ]
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="toggle scale:", showarrow=False,
                             x=0, y=1.08, yref="paper", align="left")
    ]
)

# set background color
fig.update_layout(
    title="Cumulative cases by country",
    template='plotly_dark'
)

In [130]:
dtf_deaths = data.dtf_deaths.drop(['Province/State', 'Lat', 'Long'],
                       axis=1).groupby("Country/Region").sum().T[countries]

dtf_deaths

Country/Region,Spain,France,Germany,Russia,China,India,Switzerland,Sweden,Ukraine,Colombia,Croatia,Lithuania,Taiwan*
1/22/20,0,0,0,0,17,0,0,0,0,0,0,0,0
1/23/20,0,0,0,0,18,0,0,0,0,0,0,0,0
1/24/20,0,0,0,0,26,0,0,0,0,0,0,0,0
1/25/20,0,0,0,0,42,0,0,0,0,0,0,0,0
1/26/20,0,0,0,0,56,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5/9/20,26478,26313,7549,1827,4637,2101,1830,3220,376,445,87,49,6
5/10/20,26621,26383,7569,1915,4637,2212,1833,3225,391,463,90,50,6
5/11/20,26744,26646,7661,2009,4637,2294,1845,3256,408,479,91,50,7
5/12/20,26920,26994,7738,2116,4637,2415,1867,3313,425,493,91,50,7


In [131]:
fig = go.Figure()

for idx, country in enumerate(countries):
    fig.add_trace(go.Scatter(
        x=dtf_deaths.index,
        y=dtf_deaths[country],
        name=country,
        mode='lines'
    ))

# add linear/log toggle button
fig.update_layout(    
    updatemenus = [
        dict(type="buttons",
             showactive=False,
             direction="right",
             buttons=list([            
                dict(label='lin/log',
                    method='relayout',
                    args=[{'yaxis': {'type': 'linear'}}],
                    args2=[{'yaxis': {'type': 'log'}}])
            ]),
            x=0.17,
            xanchor="left",
            y=1.095,
            yanchor="top"
        )
    ]
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="toggle scale:", showarrow=False,
                             x=0, y=1.08, yref="paper", align="left")
    ]
)

# set background color
fig.update_layout(
    title="Cumulative deaths by country",
    template='plotly_dark'
)

In [132]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=countries,
    y=100*dtf_deaths[-1:].values[0]/dtf_cases[-1:].values[0]
))

# set background color
fig.update_layout(
    title="Mortality in % by country",
    template='plotly_dark',
    yaxis = dict(ticksuffix = '% ')
)

### Find the 10 most active countries

In [133]:
dtf_all = data.dtf_cases.drop(['Province/State', 'Lat', 'Long'], axis=1).groupby("Country/Region").sum().T

daily = (dtf_all - dtf_all.shift(1)).T 

countries = list(daily.sort_values(by=daily.columns[-1], ascending=False).index[:10])

countries

['US',
 'Brazil',
 'Russia',
 'Peru',
 'India',
 'United Kingdom',
 'Chile',
 'Iran',
 'Saudi Arabia',
 'Mexico']

In [134]:
dtf_cases = data.dtf_cases.drop(['Province/State', 'Lat', 'Long'], axis=1).groupby("Country/Region").sum().T[countries]

dtf_cases

Country/Region,US,Brazil,Russia,Peru,India,United Kingdom,Chile,Iran,Saudi Arabia,Mexico
1/22/20,1,0,0,0,0,0,0,0,0,0
1/23/20,1,0,0,0,0,0,0,0,0,0
1/24/20,2,0,0,0,0,0,0,0,0,0
1/25/20,2,0,0,0,0,0,0,0,0,0
1/26/20,5,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
5/9/20,1309550,156061,198676,65015,62808,216525,27219,106220,37136,33460
5/10/20,1329260,162699,209688,67307,67161,220449,28866,107603,39048,35022
5/11/20,1347881,169594,221344,68822,70768,224332,30063,109286,41014,36327
5/12/20,1369376,178214,232243,72059,74292,227741,31721,110767,42925,38324


In [135]:
fig = go.Figure()

for idx, country in enumerate(countries):
    fig.add_trace(go.Scatter(
        x=dtf_cases.index,
        y=dtf_cases[country],
        name=country
    ))

# add linear/log toggle button
fig.update_layout(    
    updatemenus = [
        dict(type="buttons",
             showactive=False,
             direction="right",
             buttons=list([            
                dict(label='lin/log',
                    method='relayout',
                    args=[{'yaxis': {'type': 'linear'}}],
                    args2=[{'yaxis': {'type': 'log'}}])
            ]),
            x=0.175,
            xanchor="left",
            y=1.095,
            yanchor="top"
        )
    ]
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="toggle scale:", showarrow=False,
                             x=0, y=1.08, yref="paper", align="left")
    ]
)

# set background color
fig.update_layout(
    title="Cumulative cases by country",
    template='plotly_dark'
)

In [136]:
dtf_deaths = data.dtf_deaths.drop(['Province/State', 'Lat', 'Long'], axis=1).groupby("Country/Region").sum().T[countries]

dtf_deaths

Country/Region,US,Brazil,Russia,Peru,India,United Kingdom,Chile,Iran,Saudi Arabia,Mexico
1/22/20,0,0,0,0,0,0,0,0,0,0
1/23/20,0,0,0,0,0,0,0,0,0,0
1/24/20,0,0,0,0,0,0,0,0,0,0
1/25/20,0,0,0,0,0,0,0,0,0,0
1/26/20,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
5/9/20,78795,10656,1827,1814,2101,31662,304,6589,239,3353
5/10/20,79526,11123,1915,1889,2212,31930,312,6640,246,3465
5/11/20,80682,11653,2009,1961,2294,32141,323,6685,255,3573
5/12/20,82356,12461,2116,2057,2415,32769,335,6733,264,3926


In [137]:
fig = go.Figure()

for idx, country in enumerate(countries):
    fig.add_trace(go.Scatter(
        x=dtf_deaths.index,
        y=dtf_deaths[country],
        name=country,
        mode='lines'
    ))

# add linear/log toggle button
fig.update_layout(    
    updatemenus = [
        dict(type="buttons",
             showactive=False,
             direction="right",
             buttons=list([            
                dict(label='lin/log',
                    method='relayout',
                    args=[{'yaxis': {'type': 'linear'}}],
                    args2=[{'yaxis': {'type': 'log'}}])
            ]),
            x=0.17,
            xanchor="left",
            y=1.095,
            yanchor="top"
        )
    ]
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="toggle scale:", showarrow=False,
                             x=0, y=1.08, yref="paper", align="left")
    ]
)

# set background color
fig.update_layout(
    title="Cumulative deaths by country",
    template='plotly_dark'
)

In [138]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=countries,
    y=100*dtf_deaths.iloc[-1]/dtf_cases.iloc[-1]
))

# set background color
fig.update_layout(
    title="Mortality in % by country",
    template='plotly_dark',
    yaxis = dict(ticksuffix = '% ')
)

### Worldmap showing cummulated cases today

In [139]:
dtf_all_cases = data.dtf_cases.drop(['Province/State', 'Lat', 'Long'], axis=1).groupby("Country/Region").sum()
dtf_all_cases.columns = pd.to_datetime(dtf_all_cases.columns, format='%Y-%m-%d', infer_datetime_format=True)
dtf_all_cases.index.name="Country"

In [140]:
dtf_all_cases

,2020-01-22,2020-01-23,2020-01-24,2020-01-25,2020-01-26,2020-01-27,2020-01-28,2020-01-29,2020-01-30,2020-01-31,...,2020-05-04,2020-05-05,2020-05-06,2020-05-07,2020-05-08,2020-05-09,2020-05-10,2020-05-11,2020-05-12,2020-05-13
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,0,0,0,0,0,0,0,0,0,0,...,2894,3224,3392,3563,3778,4033,4402,4687,4963,5226
Albania,0,0,0,0,0,0,0,0,0,0,...,803,820,832,842,850,856,868,872,876,880
Algeria,0,0,0,0,0,0,0,0,0,0,...,4648,4838,4997,5182,5369,5558,5723,5891,6067,6253
Andorra,0,0,0,0,0,0,0,0,0,0,...,750,751,751,752,752,754,755,755,758,760
Angola,0,0,0,0,0,0,0,0,0,0,...,35,36,36,36,43,43,45,45,45,45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
West Bank and Gaza,0,0,0,0,0,0,0,0,0,0,...,362,371,374,375,375,375,375,375,375,375
Western Sahara,0,0,0,0,0,0,0,0,0,0,...,6,6,6,6,6,6,6,6,6,6
Yemen,0,0,0,0,0,0,0,0,0,0,...,12,22,25,25,34,34,51,56,65,70


In [141]:
fig = go.Figure(data=go.Choropleth(
    locationmode = "country names",
    locations = dtf_all_cases.index,
    z = dtf_all_cases[dtf_all_cases.columns[-1]],
    text = dtf_all_cases.index,
    colorscale = 'Aggrnyl',
    reversescale=True,
    autocolorscale=False,
    marker_line_color='darkgray',
    marker_line_width=0.5
))

fig.update_layout(
    title_text='Cummulated cases worldwide',
    template='plotly_dark',
    width=700,
    geo=dict(
        projection_type='equirectangular'
    ),
    annotations = [dict(
        x=0,
        y=-0.1,
        xref='paper',
        yref='paper',
        text='Source: <a href="https://github.com/CSSEGISandData/COVID-19">Johns Hopkins CSSE</a>',
        showarrow = False
    )]
)

In [142]:
from ipywidgets import interact

max_days = len(dtf_all_cases.columns)-1

fig_wg = go.FigureWidget(fig)

fig_wg.update_layout(title_text='Cummulated cases worldwide ' + dtf_all_cases.columns[max_days].strftime('%d/%m/%Y'))

@interact(day=(0, max_days, 1))
def update(day=max_days):
    date = dtf_all_cases.columns[day]
    fig_wg.update_layout(title_text='Cummulated cases worldwide ' + date.strftime('%d/%m/%Y'))
    with fig_wg.batch_update():
        fig_wg.data[0].z = dtf_all_cases[date]

fig_wg
        

interactive(children=(IntSlider(value=112, description='day', max=112), Output()), _dom_classes=('widget-inter…

FigureWidget({
    'data': [{'autocolorscale': False,
              'colorscale': [[0.0, 'rgb(36, 86, 104)'], …